# GloveSpeak — Gesture Recognition Training
## Realtime BISINDO untuk percakapan dua arah
### Perbaikan: sliding window · delta features · adaptive segmenter · per-kategori window

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')

# Import semua komponen dari file yang sudah diperbaiki
from advanced_gesture_recognition import (
    build_bilstm_attention_model,
    build_tcn_model,
    GloveSensorPreprocessor,
    AdaptiveGestureSegmenter,
    GrammarPostprocessor,
    RealtimeInferenceEngine,
    ProductionInferenceEngine,
    AttentionLayer,
    CATEGORY_WINDOW,
    CONFIDENCE_THRESHOLD,
    NUM_TOTAL_FEATURES,
    SAMPLING_RATE,
)

print("TensorFlow:", tf.__version__)
print("GPU:", len(tf.config.list_physical_devices('GPU')) > 0)
print("Features per frame:", NUM_TOTAL_FEATURES, "(raw 22 + delta 22 + accel 22)")
print("Window sizes:", CATEGORY_WINDOW)


## 3.5 — Augmentasi Data (5 rep → 45 sampel per gesture)

In [ ]:
# Import augmentor
from sensor_augmentation import SensorDataAugmenter, validate_augmentation

print(f"Data SEBELUM augmentasi: {len(X_raw)} sampel, {len(set(y_raw))} gesture")
print(f"Rata-rata per gesture: {len(X_raw)/max(len(set(y_raw)),1):.1f}")
print()

# Augmentasi — 8 teknik, menghasilkan 9x data (original + 8 augmented)
augmenter = SensorDataAugmenter(seed=42)

# Gunakan augment_balanced agar setiap gesture punya minimal 45 sampel
# (berguna jika ada gesture dengan < 5 rekaman)
X_aug_raw, y_aug_raw = augmenter.augment_balanced(
    X_raw, y_raw,
    target_per_class=45,
    verbose=True,
)

print(f"\nData SETELAH augmentasi: {len(X_aug_raw)} sampel")
print(f"Rata-rata per gesture: {len(X_aug_raw)/max(len(set(y_aug_raw)),1):.1f}")

# Validasi visual — cek augmentasi tidak merusak sinyal
if len(X_raw) > 0:
    sample = X_raw[0]
    aug_samples = augmenter.augment_one(sample)
    validate_augmentation([sample], aug_samples, sample_idx=0, feature_idx=5)


## 1. Load Gesture List

In [ ]:
def load_gesture_list(filename='bisindo_gesture_list.txt'):
    """
    Load gesture list dari file.
    Format: CATEGORY,gesture_label
    Return: list gesture labels, dict category->indices
    """
    gestures = []
    categories = {}
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split(',', 1)
            if len(parts) == 2:
                cat, label = parts[0].strip(), parts[1].strip()
                idx = len(gestures)
                gestures.append(label)
                categories.setdefault(cat, []).append(idx)
    return gestures, categories

gestures, categories = load_gesture_list()
num_gestures = len(gestures)

print(f"Total gestures: {num_gestures}")
for cat, idxs in categories.items():
    print(f"  {cat}: {len(idxs)} gestures")
print()
print("Contoh:")
for cat, idxs in categories.items():
    sample = gestures[idxs[0]]
    print(f"  {cat}[0] = '{sample}'")


## 2. Load Data (per-kategori subfolder)

In [ ]:
def load_data_from_folders(base_path='datashet', gestures=None, categories=None):
    """
    Load semua CSV dari struktur folder:
      datashet/
        angka/    ← kategori lowercase
        huruf/
        kata/
        frasa/

    Setiap CSV: kolom timestamp + 22 sensor kolom + repetition.
    Return: X_raw (list of np.array (T,22)), y (list of int label idx)
    """
    X_raw, y = [], []

    # Buat mapping label -> index
    label_to_idx = {g: i for i, g in enumerate(gestures)}

    # Scan setiap kategori
    for cat, cat_idxs in categories.items():
        cat_dir = os.path.join(base_path, cat.lower())
        if not os.path.exists(cat_dir):
            print(f"  [WARN] Folder tidak ditemukan: {cat_dir}")
            continue

        cat_count = 0
        for csv_file in Path(cat_dir).glob('*.csv'):
            try:
                df = pd.read_csv(csv_file)

                # Kolom sensor: semua kecuali timestamp dan repetition
                drop_cols = [c for c in ['timestamp', 'repetition'] if c in df.columns]
                sensor_df = df.drop(columns=drop_cols)

                if sensor_df.shape[1] != 22:
                    print(f"  [SKIP] {csv_file.name}: {sensor_df.shape[1]} kolom (expected 22)")
                    continue

                data = sensor_df.values.astype(np.float32)
                if len(data) == 0:
                    continue

                # Tentukan label dari nama file (format: label_rep1_timestamp.csv)
                # Coba cocokkan dengan gesture list dari kategori ini
                fname = csv_file.stem  # tanpa .csv
                # ambil bagian sebelum _rep
                if '_rep' in fname:
                    raw_label = fname[:fname.index('_rep')].replace('_', ' ')
                else:
                    raw_label = fname.replace('_', ' ')

                if raw_label in label_to_idx:
                    label_idx = label_to_idx[raw_label]
                else:
                    # Coba partial match
                    matched = [g for g in gestures if g in raw_label or raw_label in g]
                    if matched:
                        label_idx = label_to_idx[matched[0]]
                    else:
                        print(f"  [SKIP] Tidak cocok: '{raw_label}' dari {csv_file.name}")
                        continue

                X_raw.append(data)
                y.append(label_idx)
                cat_count += 1

            except Exception as e:
                print(f"  [ERROR] {csv_file.name}: {e}")

        print(f"  {cat}: {cat_count} recordings dimuat")

    print(f"\nTotal: {len(X_raw)} recordings, {len(set(y))} unique gestures")
    return X_raw, y

X_raw, y_raw = load_data_from_folders('datashet', gestures, categories)

if len(X_raw) == 0:
    print("\n[STOP] Tidak ada data ditemukan!")
    print("Pastikan folder 'datashet/angka/', 'datashet/huruf/', dll. ada dan berisi CSV.")
else:
    seq_lengths = [len(s) for s in X_raw]
    print(f"Panjang sequence: min={min(seq_lengths)}, max={max(seq_lengths)}, median={np.median(seq_lengths):.0f} frame")
    y_arr = np.array(y_raw)
    counts = np.bincount(y_arr, minlength=num_gestures)
    print(f"Rata-rata sampel per gesture: {counts[counts>0].mean():.1f}")
    print(f"Gesture belum ada data: {(counts==0).sum()}")


## 3. Preprocessing — 66 fitur, post-padding

In [ ]:
# Pilih window size (gunakan 'ALL' untuk satu model universal)
WINDOW_SIZE = CATEGORY_WINDOW['ALL']  # 80 frame = 800ms

# Fit preprocessor dari data training
preprocessor = GloveSensorPreprocessor()
preprocessor.fit(X_aug_raw)

print(f"Preprocessing {len(X_raw)} sequences ke window={WINDOW_SIZE} frame...")
print(f"Fitur output: {NUM_TOTAL_FEATURES} (raw 22 + delta 22 + accel 22)")

X_processed = preprocessor.batch_transform(X_aug_raw, WINDOW_SIZE)
y_processed  = np.array(y_aug_raw)

print(f"\nShape X: {X_processed.shape}")
print(f"Shape y: {y_processed.shape}")
print(f"Mean  X: {X_processed.mean():.4f}")
print(f"Std   X: {X_processed.std():.4f}")

# Simpan metadata untuk inference (Android / testing)
metadata = {
    'window_size'           : WINDOW_SIZE,
    'num_gestures'          : num_gestures,
    'num_features'          : NUM_TOTAL_FEATURES,
    'sampling_rate'         : SAMPLING_RATE,
    'confidence_threshold'  : CONFIDENCE_THRESHOLD,
    'scaler'                : preprocessor.to_dict(),
    'gesture_labels'        : gestures,
    'category_windows'      : CATEGORY_WINDOW,
}
with open('model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("\nMetadata disimpan: model_metadata.json")


In [ ]:
# Train/Validation split — stratified agar tiap gesture terwakili
X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y_processed,
    test_size=0.2, random_state=42, stratify=y_processed
)
print(f"Train: {X_train.shape}  |  Val: {X_val.shape}")
print(f"Label unik train: {len(np.unique(y_train))} | val: {len(np.unique(y_val))}")


## 4. Build Model

In [ ]:
# ── Pilihan model ─────────────────────────────────────────────────────────
# Ganti MODEL_TYPE ke 'TCN' untuk model lebih cepat (~3x) tapi sedikit kurang akurat
MODEL_TYPE = 'BiLSTM'   # 'BiLSTM' | 'TCN'

if MODEL_TYPE == 'BiLSTM':
    model = build_bilstm_attention_model(
        num_gestures=num_gestures,
        window_size=WINDOW_SIZE,
        num_features=NUM_TOTAL_FEATURES,
        lstm_units=128,
        dense_units=128,
        dropout_rate=0.35,
    )
else:
    model = build_tcn_model(
        num_gestures=num_gestures,
        window_size=WINDOW_SIZE,
        num_features=NUM_TOTAL_FEATURES,
        filters=64,
        kernel_size=3,
        dropout_rate=0.2,
    )

model.summary()
print(f"\nModel: {MODEL_TYPE}")
print(f"Parameter: {model.count_params():,}")


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
print("Compile OK")


## 5. Training

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7,
        min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_gesture_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=120,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)
print("\nTraining selesai!")


## 6. Evaluasi

In [ ]:
# Load model terbaik (sudah di-restore oleh EarlyStopping)
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss,   val_acc   = model.evaluate(X_val,   y_val,   verbose=0)
print(f"Train  acc: {train_acc*100:.2f}%  loss: {train_loss:.4f}")
print(f"Val    acc: {val_acc*100:.2f}%  loss: {val_loss:.4f}")


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set(title='Loss', xlabel='Epoch', ylabel='Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('training_history.png', dpi=100); plt.show()


In [ ]:
# Classification report per gesture
y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
present_labels = sorted(np.unique(np.concatenate([y_val, y_pred])))
target_names   = [gestures[i] for i in present_labels]
print(classification_report(y_val, y_pred, labels=present_labels, target_names=target_names, digits=3))

# Per-gesture accuracy — sort untuk lihat yang paling lemah
accs = {}
for idx in present_labels:
    mask = y_val == idx
    if mask.sum() > 0:
        accs[gestures[idx]] = (y_pred[mask] == idx).mean()

sorted_accs = sorted(accs.items(), key=lambda x: x[1])
print("\n10 Gesture TERLEMAH:")
for g, a in sorted_accs[:10]:
    bar = '█' * int(a * 20)
    print(f"  {g:25s} {a*100:5.1f}%  {bar}")


In [ ]:
# Confusion matrix — top 30 gesture (agar mudah dibaca)
from sklearn.metrics import confusion_matrix

TOP_N = min(30, len(present_labels))
top_labels = sorted(accs.items(), key=lambda x: x[1])[:TOP_N]
top_idxs   = [present_labels[list(accs.keys()).index(g)] for g, _ in top_labels]

cm = confusion_matrix(y_val, y_pred, labels=top_idxs)
top_names = [gestures[i] for i in top_idxs]

plt.figure(figsize=(14, 12))
sns.heatmap(cm, xticklabels=top_names, yticklabels=top_names,
            fmt='d', cmap='Blues', cbar=False, linewidths=0.3)
plt.title(f'Confusion Matrix — {TOP_N} gesture terlemah')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=100); plt.show()


## 7. Optimasi Confidence Threshold

In [ ]:
# Cari threshold optimal: tinggi = lebih sedikit output tapi lebih akurat
# Ini penting untuk mencegah false positive saat tangan diam

y_probs = model.predict(X_val, verbose=0)  # (N, num_gestures)

results = []
for thr in np.arange(0.50, 0.95, 0.025):
    mask    = y_probs.max(axis=1) >= thr
    covered = mask.sum() / len(mask)        # % data yang melewati threshold
    if mask.sum() == 0:
        continue
    acc = (y_probs[mask].argmax(axis=1) == y_val[mask]).mean()
    results.append({'threshold': thr, 'coverage': covered, 'accuracy': acc})

res_df = pd.DataFrame(results)

# Plot
fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.plot(res_df['threshold'], res_df['accuracy'] * 100, 'b-o', label='Accuracy (%)', markersize=4)
ax2.plot(res_df['threshold'], res_df['coverage'] * 100, 'r--s', label='Coverage (%)', markersize=4)
ax1.set_xlabel('Confidence Threshold'); ax1.set_ylabel('Accuracy (%)', color='b'); ax2.set_ylabel('Coverage (%)', color='r')
ax1.set_title('Accuracy vs Coverage pada berbagai threshold')
ax1.grid(alpha=0.3)
ax1.axvline(CONFIDENCE_THRESHOLD, color='green', linestyle=':', label=f'Default ({CONFIDENCE_THRESHOLD})')
fig.legend(loc='lower right', bbox_to_anchor=(0.88, 0.15)); plt.tight_layout(); plt.show()

# Threshold terbaik: accuracy >= 85% dengan coverage semaksimal mungkin
good = res_df[res_df['accuracy'] >= 0.85]
if len(good) > 0:
    optimal = good.iloc[0]
    print(f"Threshold optimal: {optimal['threshold']:.3f}  "
          f"acc={optimal['accuracy']*100:.1f}%  coverage={optimal['coverage']*100:.1f}%")
    # Update metadata
    metadata['confidence_threshold'] = float(optimal['threshold'])
    with open('model_metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    print("Threshold diperbarui di model_metadata.json")
else:
    print("Tidak ada threshold yang mencapai acc 85% — tambah data atau latih lebih lama")
print(res_df.to_string(index=False))


## 8. Convert ke TFLite untuk Android

In [ ]:
# ── Konversi dengan quantization INT8 (ukuran ~4x lebih kecil) ──────────
# INT8 representative dataset
def representative_dataset():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

# Standard TFLite (float32) — baseline
converter_f32 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_f32.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_f32 = converter_f32.convert()
with open('gesture_model_f32.tflite', 'wb') as f: f.write(tflite_f32)
print(f"Float32 TFLite: {len(tflite_f32)/1024:.1f} KB")

# INT8 quantized — untuk Android mid-range (lebih cepat ~2x)
converter_i8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_i8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_i8.representative_dataset = representative_dataset
converter_i8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_i8.inference_input_type  = tf.float32   # input tetap float32 untuk kemudahan
converter_i8.inference_output_type = tf.float32   # output tetap float32
tflite_i8 = converter_i8.convert()
with open('gesture_model_int8.tflite', 'wb') as f: f.write(tflite_i8)
print(f"INT8 TFLite   : {len(tflite_i8)/1024:.1f} KB")
print(f"Kompresi      : {len(tflite_f32)/len(tflite_i8):.1f}x")


In [ ]:
# ── Ukur latensi inferensi TFLite (simulasi kondisi Android) ──────────
import time

def benchmark_tflite(model_path, X_test, n_runs=100):
    interp = tf.lite.Interpreter(model_path=model_path)
    interp.allocate_tensors()
    inp_idx = interp.get_input_details()[0]['index']
    out_idx = interp.get_output_details()[0]['index']

    times = []
    for i in range(n_runs):
        sample = X_test[i % len(X_test):i % len(X_test)+1].astype(np.float32)
        t0 = time.perf_counter()
        interp.set_tensor(inp_idx, sample)
        interp.invoke()
        _ = interp.get_tensor(out_idx)
        times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    return {'mean_ms': times.mean(), 'p95_ms': np.percentile(times, 95), 'min_ms': times.min()}

for mpath in ['gesture_model_f32.tflite', 'gesture_model_int8.tflite']:
    if os.path.exists(mpath):
        b = benchmark_tflite(mpath, X_val)
        label = 'Float32' if 'f32' in mpath else 'INT8  '
        realtime_ok = '✓ OK' if b['mean_ms'] < 150 else '✗ LAMBAT'
        print(f"{label}: mean={b['mean_ms']:.1f}ms  p95={b['p95_ms']:.1f}ms  {realtime_ok}")
        print(f"         (Stride 150ms = inferensi harus < 150ms untuk realtime)")


In [ ]:
# Pastikan accuracy TFLite sama dengan Keras (max loss 0.5%)
interp = tf.lite.Interpreter(model_path='gesture_model_int8.tflite')
interp.allocate_tensors()
inp_idx = interp.get_input_details()[0]['index']
out_idx = interp.get_output_details()[0]['index']

y_pred_tflite = []
for i in range(len(X_val)):
    interp.set_tensor(inp_idx, X_val[i:i+1].astype(np.float32))
    interp.invoke()
    out = interp.get_tensor(out_idx)[0]
    y_pred_tflite.append(np.argmax(out))

tflite_acc = (np.array(y_pred_tflite) == y_val).mean()
keras_acc   = (model.predict(X_val, verbose=0).argmax(axis=1) == y_val).mean()
delta = abs(keras_acc - tflite_acc)

print(f"Keras  accuracy: {keras_acc*100:.2f}%")
print(f"TFLite accuracy: {tflite_acc*100:.2f}%")
print(f"Delta           : {delta*100:.2f}% {'✓ OK' if delta < 0.005 else '✗ Cek quantization'}")


## 9. Simulasi Realtime Inference (Sliding Window)

In [ ]:
# Simulasikan bagaimana data dari ESP32 akan diproses
# Ambil 1 sequence val, pura-pura datang frame per frame

test_idx = 0
test_seq_raw   = X_raw[test_idx]    # (T, 22) — raw
test_seq_label = y_raw[test_idx]
print(f"Sequence: '{gestures[test_seq_label]}' ({len(test_seq_raw)} frame)")

# Buat inference engine
segmenter_rt = AdaptiveGestureSegmenter()
# Kalibrasi: ambil 30 frame pertama sebagai baseline 'diam'
if len(test_seq_raw) > 30:
    segmenter_rt.calibrate(np.array(test_seq_raw[:30]))

detected_gestures = []

def on_gesture(result):
    detected_gestures.append(result)
    print(f"  [{result['timestamp_ms']:5d}ms] GESTURE: {result['gesture']:20s}  "
          f"conf={result['confidence']:.2f}  stable={result['stable']}")

engine = RealtimeInferenceEngine(
    model=model,
    preprocessor=preprocessor,
    segmenter=segmenter_rt,
    gesture_labels=gestures,
    window_size=WINDOW_SIZE,
    on_gesture_callback=on_gesture,
)

# Simulasi: push frame satu-satu seperti dari UDP ESP32
print("\nSimulasi stream realtime:")
for t, frame in enumerate(test_seq_raw):
    ts_ms = t * (1000 // SAMPLING_RATE)  # timestamp dalam ms
    engine.push_frame(np.array(frame), timestamp_ms=ts_ms)

print(f"\nTotal gesture terdeteksi: {len(detected_gestures)}")
if detected_gestures:
    grammar = GrammarPostprocessor()
    result = grammar.build_tts_output(detected_gestures)
    print(f"Kalimat TTS: '{result['sentence']}'")
    print(f"Confidence rata-rata: {result['confidence']:.2f}")


## 10. Simpan Semua Artifacts

In [ ]:
# Simpan Keras model
model.save('best_gesture_model.keras')
print("✓ best_gesture_model.keras")

# Metadata sudah disimpan di Cell 4 & 7
print("✓ model_metadata.json")

# TFLite sudah disimpan di Cell 8
print("✓ gesture_model_f32.tflite")
print("✓ gesture_model_int8.tflite  ← pakai ini untuk Android")

print("""
╔══════════════════════════════════════════════════════════════╗
║  Artifacts siap untuk Android deployment:                    ║
║  1. gesture_model_int8.tflite  — model utama                 ║
║  2. model_metadata.json        — scaler + threshold + labels ║
║     field penting:                                           ║
║       window_size, num_features, confidence_threshold        ║
║       scaler.mean, scaler.scale  (untuk preprocessing)       ║
║       gesture_labels             (150 label BISINDO)         ║
║  3. best_gesture_model.keras   — backup full precision       ║
╚══════════════════════════════════════════════════════════════╝
""")
